In [ ]:
from src.data import Dataloader
import pandas as pd
import os
from src.utils import seed_everything
import random
from tqdm import tqdm
from typing import Callable
import torch.nn.functional as F
from src.data import TimeSeriesDataset
import torch
from src.nn import MLP, SlidingWindowBinaryClassification, SlidingWindowRegression
from sklearn.metrics import roc_auc_score, precision_score
import warnings
from sklearn.exceptions import DataConversionWarning

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

warnings.filterwarnings("ignore", category=DataConversionWarning)


## 1. Set up
To set up experience environment, we perform the following steps:

1. This section below will set up `const` needed for experiment
2. For reproducibility, we also set the same inital seed for everything (`numpy`, `torch`)

In [2]:
# Step 1.
DATA_PATH = os.path.abspath('data/clean/')
VAL_START_DATE = int(pd.Timestamp('2023-12-20').timestamp())  # Unix time in seconds
TEST_START_DATE = int(pd.Timestamp('2024-12-20').timestamp()) # Unix time in seconds
FEAT_COLUMN = ['Close','High','Low', 'Open']
BINARY_LABEL_COLUMN = 'price_increase'
REGRESSION_LABEL_COLUMN = 'next_close'
LOG_SCALING = [2,4,8,16,32]

INIT_SEED = 720

# Step 2.
seed_everything(INIT_SEED)

## 2. Prepare dataset for training

1. Shuffle all token networks (I.I.D)
2. Split datasets for training, validation and testing: the first 70% of datasets for training, the next 15% for validation, the next 15% for testing
3. Load all dataset in memory to `TimeSeriesDataset`
4. For each token, we use all data before `2023-12-20` for trainning, from `2023-12-20` to `2024-12-20` for validation and `2024-12-20` onward for testing

There are 49 tokens datasets in total, so we keep 32 networks for training, 17 tokens for validation and 16 tokens for testing.

In [3]:
files = os.listdir(DATA_PATH)
random.shuffle(files) # Step 1

# Step 2
train_token_list = files[:32]
valid_token_list = files[32:33+8]
test_token_list = files [-8:]

assert len(set(train_token_list).intersection(set(valid_token_list))) == 0
assert len(set(test_token_list).intersection(set(valid_token_list))) == 0

# Step 3
data_loader = Dataloader(DATA_PATH)

train_data = []
valid_data = []
test_data = []

for file_name in tqdm(train_token_list):
    train_data.append(data_loader.from_csv(file_name, feat_columns=FEAT_COLUMN, label_column=BINARY_LABEL_COLUMN))

for file_name in tqdm(valid_token_list):
    valid_data.append(data_loader.from_csv(file_name, feat_columns=FEAT_COLUMN, label_column=BINARY_LABEL_COLUMN))

for file_name in tqdm(test_token_list):
    test_data.append(data_loader.from_csv(file_name, feat_columns=FEAT_COLUMN, label_column=BINARY_LABEL_COLUMN))

train_data_split = []
valid_data_split = []
test_data_split = []

for data in tqdm(train_data):
    train, val_test = data.split(VAL_START_DATE)
    val, test = val_test.split(TEST_START_DATE)
    train_data_split.append((train,val,test))

for data in tqdm(valid_data):
    train, val_test = data.split(VAL_START_DATE)
    val, test = val_test.split(TEST_START_DATE)
    valid_data_split.append((train,val,test))

for data in tqdm(test_data):
    train, val_test = data.split(VAL_START_DATE)
    val, test = val_test.split(TEST_START_DATE)
    test_data_split.append((train,val,test))

100%|██████████| 8/8 [00:00<00:00, 1037.36it/s]


4. Next we normalize each feature with min and max of each feature

In [4]:
def normalize(train_data: TimeSeriesDataset, valid_data: TimeSeriesDataset, test_data: TimeSeriesDataset, normalize_y = False):
    
    max_values,_ = torch.max(train_data.x, dim=0)
    min_values,_ = torch.min(train_data.x, dim=0)

    train_data.x = (train_data.x - min_values) / (max_values - min_values)
    valid_data.x = (valid_data.x - min_values) / (max_values - min_values)
    test_data.x = (test_data.x - min_values) / (max_values - min_values)

    if normalize_y:
        max_values,_ = torch.max(train_data.y, dim=0)
        min_values,_ = torch.min(train_data.y, dim=0)

        train_data.y = (train_data.y - min_values) / (max_values - min_values)
        valid_data.y = (valid_data.y - min_values) / (max_values - min_values)
        test_data.y = (test_data.y - min_values) / (max_values - min_values)

    return train_data,valid_data, test_data


for data_split in [train_data_split, valid_data_split, test_data_split]:
    for train,val, test in data_split:
        normalize(train,val,test)
        


## 3. Traditional machine learning baseline

In this project, we explore the following traditional machine learning for binary classification
1. Logistic Regression
2. Random Forest
3. Gradient Boosting (GBM)
4. SVM
5. KNN

In [12]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import roc_auc_score
import numpy as np

def traditional_model_experience(n_tokens, train_data, test_data):
    used_train_data = train_data[:n_tokens]
    all_train_x = []
    all_train_y = []
    all_test_x = []
    all_test_y = []

    for train, _, test in used_train_data:
        all_train_x.append(train.x)
        all_train_y.append(train.y)
        
    for train, _, test in test_data: 
        all_test_y.append(test.y)
        all_test_x.append(test.x)

    train_x = torch.concatenate(all_train_x).numpy()
    train_y = torch.concatenate(all_train_y).numpy()
    test_x = torch.concatenate(all_test_x).numpy()
    test_y = torch.concatenate(all_test_y).squeeze().numpy()

    models = {
        "Logistic Regression":     LogisticRegression(),
        "Random Forest":           RandomForestClassifier(),
        "Gradient Boosting (GBM)": GradientBoostingClassifier(),
        "SVM":                     SVC(probability=True),
        "KNN":                     KNeighborsClassifier(),
    }
    results = {}
    results['# Training tokens'] = n_tokens
    for name, model in tqdm(models.items()):
        model.fit(train_x, train_y)
        preds = model.predict_proba(test_x)[:, 1]
        auc = roc_auc_score(test_y, preds)
        results[name] = auc

    return results

all_result = []
for n in LOG_SCALING:
    n_result = traditional_model_experience(n,train_data_split,test_data_split[:1] )
    all_result.append(n_result)

traditional_results_df = pd.DataFrame(all_result)
traditional_results_df


100%|██████████| 5/5 [02:24<00:00, 28.93s/it]


,# Training tokens,Logistic Regression,Random Forest,Gradient Boosting (GBM),SVM,KNN
0,2,0.504993,0.527268,0.551276,0.487712,0.527074
1,4,0.494036,0.522649,0.524549,0.511179,0.522843
2,8,0.502275,0.524549,0.524119,0.556227,0.521914
3,16,0.499556,0.512677,0.500902,0.542857,0.493703
4,32,0.490680,0.514771,0.506644,0.511872,0.509320


4. Sliding window baseline

In [33]:

auc = []
for data in test_data_split:
    baseline = SlidingWindowBinaryClassification()
    train, val, test = data
    preds = []

    baseline.update(val.y[-1].item())
    for _,_,y in test:
        y = baseline()
        preds.append(y)
        baseline.update(y) 

    score = roc_auc_score(test.y.squeeze().tolist(), preds)
    auc.append(score)

sliding_window_result_df = pd.DataFrame([{'Sliding Window - Binary Classification': sum(auc)/len(auc)}])
sliding_window_result_df

,Sliding Window - Binary Classification
0,0.5


## Training loop

In [ ]:

data_loader = Dataloader(DATA_PATH)
timeseries_data = data_loader.from_csv("aave.csv", feat_columns = FEAT_COLUMN, label_column=BINARY_LABEL_COLUMN)

train_data, val_test_data = timeseries_data.split(VAL_START_DATE)
val_data, test_data = val_test_data.split(TEST_START_DATE)


train_data, val_data, test_data = normalize(train_data,val_data, test_data)
print(train_data.x.shape)


def train(train_data : TimeSeriesDataset, val_data : TimeSeriesDataset, model: torch.nn.Module, criterion : Callable, opt: torch.optim.Optimizer, epoch = 50):
    model.train()
    for i in range(epoch):
        all_loss = []
        preds = []
        for time, feat, y in train_data:
            opt.zero_grad()
            z = model(feat.float())
            loss = criterion(
                z.float(),y.float()
            )
            
            all_loss.append(loss.item())
            loss.backward()
            opt.step()

            preds.append(z.sigmoid().item())
        auc = roc_auc_score(train_data.y.squeeze().tolist(), preds)
        print(f"[INFO] Epoch - {i+1} : Loss - {sum(all_loss)/float(len(all_loss))} ; AUC : {auc}")

def evaluate(data: TimeSeriesDataset, model: torch.nn.Module, evaluator: Callable = roc_auc_score):
    model.eval()
    preds = []
    for time, feat, y in data:
        pred = model(feat).sigmoid()
        
        preds.append(pred.item())
    return evaluator(data.y.squeeze().tolist(), preds)


model = MLP(in_channel= 4, out_channel= 1, dim = 128, num_layers=3)

opt = torch.optim.Adam(
    model.parameters(), lr=float(0.001)
)

score = evaluate(test_data,model)
print(score)

train(train_data, val_data, model, F.binary_cross_entropy_with_logits,opt)

score = evaluate(test_data,model)
print(score)

           


0.5542926506290002
[INFO] Epoch - 1 : Loss - 0.6960590809993958 ; AUC : 0.4786278195488721
[INFO] Epoch - 2 : Loss - 0.6942884749703772 ; AUC : 0.49703634085213033
[INFO] Epoch - 3 : Loss - 0.6936243853073693 ; AUC : 0.506218671679198
[INFO] Epoch - 4 : Loss - 0.6930096171078306 ; AUC : 0.5154323308270677
[INFO] Epoch - 5 : Loss - 0.6923198456012263 ; AUC : 0.5216917293233083
[INFO] Epoch - 6 : Loss - 0.6917311185143319 ; AUC : 0.5272243107769423
[INFO] Epoch - 7 : Loss - 0.6916595277410276 ; AUC : 0.5281892230576442
[INFO] Epoch - 8 : Loss - 0.6915778748980154 ; AUC : 0.5275971177944863
[INFO] Epoch - 9 : Loss - 0.691253974642115 ; AUC : 0.530921052631579
[INFO] Epoch - 10 : Loss - 0.6911088466942683 ; AUC : 0.5334273182957394
[INFO] Epoch - 11 : Loss - 0.6910104204775842 ; AUC : 0.5333333333333334
[INFO] Epoch - 12 : Loss - 0.6909934980923005 ; AUC : 0.5336591478696743
[INFO] Epoch - 13 : Loss - 0.690792130252745 ; AUC : 0.5334649122807017
[INFO] Epoch - 14 : Loss - 0.690731578237273

# Regression

In [5]:
train_data = []
valid_data = []
test_data = []

for file_name in tqdm(train_token_list):
    train_data.append(data_loader.from_csv(file_name, feat_columns=FEAT_COLUMN, label_column=REGRESSION_LABEL_COLUMN))

for file_name in tqdm(valid_token_list):
    valid_data.append(data_loader.from_csv(file_name, feat_columns=FEAT_COLUMN, label_column=REGRESSION_LABEL_COLUMN))

for file_name in tqdm(test_token_list):
    test_data.append(data_loader.from_csv(file_name, feat_columns=FEAT_COLUMN, label_column=REGRESSION_LABEL_COLUMN))

train_data_split = []
valid_data_split = []
test_data_split = []

for data in tqdm(train_data):
    train, val_test = data.split(VAL_START_DATE)
    val, test = val_test.split(TEST_START_DATE)
    train_data_split.append((train,val,test))

for data in tqdm(valid_data):
    train, val_test = data.split(VAL_START_DATE)
    val, test = val_test.split(TEST_START_DATE)
    valid_data_split.append((train,val,test))

for data in tqdm(test_data):
    train, val_test = data.split(VAL_START_DATE)
    val, test = val_test.split(TEST_START_DATE)
    test_data_split.append((train,val,test))

for data_split in [train_data_split, valid_data_split, test_data_split]:
    for train,val, test in data_split:
        normalize(train,val,test,normalize_y=True)


100%|██████████| 8/8 [00:00<00:00, 1450.38it/s]


In this project, we explore the following traditional machine learning for regression
1. Linear Regression
2. Ridge
3. Lasso
4. Random Forest
5. Gradient Boosting (GBM)
6. SVR

In [ ]:

def traditional_regression_experience(n_tokens, train_data, test_data):
    used_train_data = train_data[:n_tokens]
    all_train_x = []
    all_train_y = []
    all_test_x = []
    all_test_y = []

    for train, _, test in used_train_data:
        all_train_x.append(train.x)
        all_train_y.append(train.y)  # ← change to your regression label attribute
        
    for train, _, test in test_data: 
        all_test_y.append(test.y)
        all_test_x.append(test.x)

    train_x = torch.concatenate(all_train_x).numpy()
    train_y = torch.concatenate(all_train_y).squeeze().numpy()
    test_x  = torch.concatenate(all_test_x).numpy()
    test_y  = torch.concatenate(all_test_y).squeeze().numpy()

    models = {
        "Linear Regression":       LinearRegression(),
        "Ridge":                   Ridge(alpha=1.0),
        "Lasso":                   Lasso(alpha=0.1),
        "Random Forest":           RandomForestRegressor(n_estimators=100, random_state=42),
        "Gradient Boosting (GBM)": GradientBoostingRegressor(n_estimators=100, random_state=42),
        "SVR":                     SVR(),
    }

    results = {}
    results['# Training tokens'] = n_tokens
    for name, model in tqdm(models.items()):
        model.fit(train_x, train_y)
        preds = model.predict(test_x)
        results[f"{name} RMSE"] = np.sqrt(mean_squared_error(test_y, preds))
        results[f"{name} R²"]   = r2_score(test_y, preds)

    return results

all_result = []
for n in LOG_SCALING:
    n_result = traditional_regression_experience(n, train_data_split, test_data_split[:1])
    all_result.append(n_result)

traditional_regression_df = pd.DataFrame(all_result)
traditional_regression_df

100%|██████████| 6/6 [00:23<00:00,  3.84s/it]


,# Training tokens,Linear Regression RMSE,Linear Regression R²,Ridge RMSE,Ridge R²,Lasso RMSE,Lasso R²,Random Forest RMSE,Random Forest R²,Gradient Boosting (GBM) RMSE,Gradient Boosting (GBM) R²,SVR RMSE,SVR R²
0,2,0.004103,0.974075,0.004254,0.972128,0.151525,-34.363545,0.013309,0.727174,0.014418,0.679821,0.099156,-14.143514
1,4,0.004010,0.975235,0.004049,0.974746,0.172541,-44.853276,0.013603,0.715003,0.014897,0.658203,0.098850,-14.050006
2,8,0.004619,0.967144,0.004011,0.975223,0.215627,-70.613535,0.014910,0.657600,0.017499,0.528362,0.090894,-11.724984
3,16,0.004143,0.973568,0.003947,0.976006,0.200242,-60.758354,0.013651,0.712978,0.015119,0.647932,0.097154,-13.538001
4,32,0.003924,0.976284,0.003942,0.976066,0.238749,-86.795316,0.013955,0.700045,0.015221,0.643141,0.092262,-12.110816


In [ ]:
rmse_scores = []
r2_scores = []

for data in test_data_split:
    baseline = SlidingWindowRegression()
    train, val, test = data
    preds = []

    baseline.update(val.y[-1].item())  # warm up with last val value
    for _, _, y in test:
        pred = baseline()
        preds.append(float(pred))
        baseline.update(y)

    rmse = np.sqrt(mean_squared_error(test.y.squeeze().tolist(), preds))
    r2   = r2_score(test.y.squeeze().tolist(), preds)
    rmse_scores.append(rmse)
    r2_scores.append(r2)

sliding_window_regression_df = pd.DataFrame([{
    'Sliding Window - Regression RMSE': sum(rmse_scores) / len(rmse_scores),
    'Sliding Window - Regression R²':   sum(r2_scores)   / len(r2_scores),
}])
sliding_window_regression_df

,Sliding Window - Regression RMSE,Sliding Window - Regression R²
0,0.050803,0.934662


In [11]:
# 1. Check label balance
labels = train_data.y.squeeze().tolist()
print(f"Positive rate: {sum(labels)/len(labels):.4f}")  # should be near 0.5

# 2. Check model output variance BEFORE training
model.eval()
outs = []
with torch.no_grad():
   for time, feat, y  in train_data:
        z = model(feat.float()).sigmoid().item()
        outs.append(z)

import numpy as np
print(f"Output mean: {np.mean(outs):.4f}")
print(f"Output std:  {np.std(outs):.4f}")  # near 0 = model outputs same thing for all inputs

# 3. Check feature variance after normalization
print(f"Feature mean: {train_data.x.mean(dim=0)}")
print(f"Feature std:  {train_data.x.std(dim=0)}")  # any column near 0 = dead feature

# 4. Check sizes
print(f"Train: {len(train_data)}, Val: {len(val_data)}, Test: {len(test_data)}")
print(f"Train labels: {train_data.y.shape}")
print(f"Train features: {train_data.x.shape}")


Positive rate: 0.5006
Output mean: 0.5111
Output std:  0.0685
Feature mean: tensor([0.2086, 0.1568, 0.2130, 0.2095])
Feature std:  tensor([0.2333, 0.1765, 0.2308, 0.2344])
Train: 799, Val: 366, Test: 381
Train labels: torch.Size([799, 1])
Train features: torch.Size([799, 4])
